In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from PIL import Image

2026-03-22 11:58:24.451379: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774180704.649020      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774180704.702606      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774180705.141001      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774180705.141043      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774180705.141046      55 computation_placer.cc:177] computation placer alr

In [5]:
# Healthy CT images
healthy_path = "/kaggle/input/datasets/nandeeshhu/pancrease-ct-segmenatation/images/negative"

# Tumor CT images
tumor_path = "/kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas/imagesTr"

# Image size for MobileNetV2
IMG_SIZE = (224, 224)

In [6]:
def load_images_from_folder(folder, label):
    images = []
    labels = []
    for subdir, _, files in os.walk(folder):
        for file in files:
            if file.endswith(".png") or file.endswith(".jpg") or file.endswith(".nii"):
                try:
                    # Load and resize PNG/JPG
                    if file.endswith(".nii"):
                        continue  # Skip NIfTI for now or implement nibabel
                    img_path = os.path.join(subdir, file)
                    img = Image.open(img_path).convert("RGB")
                    img = img.resize(IMG_SIZE)
                    images.append(np.array(img))
                    labels.append(label)
                except Exception as e:
                    print("Error loading:", file, e)
    return images, labels

# Load healthy and tumor images
healthy_images, healthy_labels = load_images_from_folder(healthy_path, 0)
tumor_images, tumor_labels = load_images_from_folder(tumor_path, 1)

# Combine
X = np.array(healthy_images + tumor_images)
y = np.array(healthy_labels + tumor_labels)

print("Total images:", X.shape)
print("Total labels:", y.shape)

Total images: (12060, 224, 224, 3)
Total labels: (12060,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (9648, 224, 224, 3)
Test shape: (2412, 224, 224, 3)


In [12]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow(
    X_train, y_train, batch_size=16, shuffle=True
)

test_generator = test_datagen.flow(
    X_test, y_test, batch_size=16, shuffle=False
)

In [14]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
predictions = Dense(1, activation='sigmoid')(x)  # Binary classification

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Recall()])
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [15]:
history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=15,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15


I0000 00:00:1774181238.009149     140 service.cc:152] XLA service 0x7db250003810 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774181238.009200     140 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1774181239.024559     140 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774181244.719057     140 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


603/603 ━━━━━━━━━━━━━━━━━━━━ 110s 163ms/step - accuracy: 0.9996 - loss: 0.0394 - recall: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0013 - val_recall: 0.0000e+00
Epoch 2/15
603/603 ━━━━━━━━━━━━━━━━━━━━ 88s 147ms/step - accuracy: 1.0000 - loss: 0.0012 - recall: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 4.4016e-04 - val_recall: 0.0000e+00
Epoch 3/15
603/603 ━━━━━━━━━━━━━━━━━━━━ 89s 148ms/step - accuracy: 1.0000 - loss: 4.4538e-04 - recall: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 2.1731e-04 - val_recall: 0.0000e+00
Epoch 4/15
603/603 ━━━━━━━━━━━━━━━━━━━━ 92s 152ms/step - accuracy: 1.0000 - loss: 2.3317e-04 - recall: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 1.2602e-04 - val_recall: 0.0000e+00
Epoch 5/15
603/603 ━━━━━━━━━━━━━━━━━━━━ 89s 148ms/step - accuracy: 1.0000 - loss: 1.3704e-04 - recall: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 8.0381e-05 - val_recall: 0.0000e+00
Epoch 6/15
603/603 ━━━━━━━━━━━━━━━━━━━━ 90s 149ms/step - accuracy: 1.0000 - loss: 8.9840e-05 - re